# Lab: Build a Search Engine

In this lab you build a small web search engine in six stages: clean up URLs, read the links out of a page, crawl a web, turn the crawl into a graph, rank the pages with PageRank, and answer search queries.

The web you crawl is not the internet. It is a fictional web of about one hundred pages, supplied as a dataset: a public health unit, a hospital, clinics, a pharmacy, a journal, some patient blogs, a library and a site on the history of web search. The module `miniweb` stands in for the network. `miniweb.fetch(url)` returns a status code and the page's HTML, the same two things a real web server returns.

Each stage has its own test, and each test checks only its own stage, so a problem in one stage does not cost marks in another. Most of the code is written for you. Your work is in the cells marked `TODO`.

In [ ]:
from collections import deque
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit, urlunsplit
import re

import miniweb

## Stage 0: Look around

A web browser does three things when you follow a link. It works out the full address of the page, it asks the server at that address for the page, and it reads the HTML that comes back. Run the cell below to fetch the seed page of the mini web, a hand-built directory in the style of the 1990s, and read its HTML.

In [ ]:
status, html = miniweb.fetch("https://directory.example/")
print(status)
print(html)

Status `200` means the server found the page. Ask for a page that does not exist and the server answers `404`, with no HTML.

In [ ]:
miniweb.fetch("https://directory.example/no-such-page")

## Stage 1: Normalize URLs

The same page can be written many ways. All of these name one page:

```
https://publichealth.example/topics/flu
https://publichealth.example/topics/flu/
https://publichealth.example/topics/flu#symptoms
HTTPS://PublicHealth.example/topics/flu
```

A crawler that treats them as four pages fetches the same page four times and splits its links four ways. The fix is to convert every URL to one **canonical form** before using it.

`urlsplit` breaks a URL into its parts: the scheme, the host (called `netloc`), the path, the query and the fragment. `urlunsplit` puts the parts back together.

In [ ]:
urlsplit("HTTPS://PublicHealth.example/topics/flu/?lang=fr#symptoms")

Write `normalize(url)`. It returns the canonical form of an absolute URL:

1. Lower-case the scheme and the host. Do not change the case of the path.
2. Drop the fragment (the part after `#`). It names a place within a page, not a different page.
3. If the path is empty, use `"/"`. Otherwise remove a trailing `"/"` from the path.
4. Keep the query string. `find-a-doctor?dept=cardiology` and `find-a-doctor?dept=oncology` are different pages.

In [ ]:
def normalize(url):
    parts = urlsplit(url)
    scheme = parts.scheme.lower()
    host = parts.netloc.lower()
    path = parts.path or "/"
    if path != "/" and path.endswith("/"):
        path = path.rstrip("/") or "/"
    return urlunsplit((scheme, host, path, parts.query, ""))

In [ ]:
for u in ["https://publichealth.example/topics/flu/",
          "HTTPS://PublicHealth.example/topics/flu#symptoms",
          "https://maplevalleyhospital.example",
          "https://maplevalleyhospital.example/find-a-doctor?dept=cardiology"]:
    print(u, "->", normalize(u))

## Stage 2: Extract the links from a page

A link in HTML is an `<a>` tag, and its `href` attribute gives the address. Many `href` values are **relative**: `/visiting` means "the page `/visiting` on this same site", and `../find-a-doctor` means "go up one directory". `urljoin(base, href)` resolves a relative `href` against the address of the page it appears on, as a browser does.

Reading HTML correctly is harder than it looks, so the `LinkParser` class below is written for you. It collects every `href` on a page, in order.

In [ ]:
class LinkParser(HTMLParser):
    """Collects the href of every <a> tag, and the page's visible text."""

    def __init__(self):
        super().__init__()
        self.hrefs = []
        self.text = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for name, value in attrs:
                if name == "href" and value:
                    self.hrefs.append(value)

    def handle_data(self, data):
        self.text.append(data)


def raw_hrefs(html):
    parser = LinkParser()
    parser.feed(html)
    return parser.hrefs


def extract_text(html):
    parser = LinkParser()
    parser.feed(html)
    return " ".join(parser.text)

Write `extract_links(base_url, html)`. It returns a list of the page's links, in the order they appear on the page, with duplicates kept:

1. Resolve each `href` against `base_url` with `urljoin`.
2. Normalize the result with your `normalize`.
3. Keep only links whose scheme is `http` or `https`. Links such as `mailto:` and `tel:` are not web pages.

In [ ]:
def extract_links(base_url, html):
    links = []
    for href in raw_hrefs(html):
        url = normalize(urljoin(base_url, href))
        if urlsplit(url).scheme in ("http", "https"):
            links.append(url)
    return links

In [ ]:
status, html = miniweb.fetch("https://maplevalleyhospital.example/departments/cardiology")
print(raw_hrefs(html))
print(extract_links("https://maplevalleyhospital.example/departments/cardiology", html))

## Stage 3: Crawl

A crawler starts from one page, the **seed**, and follows links until it has visited every page it can reach. This one is **breadth-first**: it keeps a queue of pages to visit, takes pages from the front, and adds newly found links to the back. A set called `seen` records every URL already queued, so that a cycle of links (A links to B, B links back to A) does not loop forever.

Complete the loop in `crawl(seed)`. For each URL taken from the queue:

1. Fetch it with `miniweb.fetch`. If the status is not `200`, skip it.
2. Store its HTML in `pages`.
3. For each link that `extract_links` finds on it, add the link to `seen` and to the queue, unless it is already in `seen`.

In [ ]:
def crawl(seed):
    seed = normalize(seed)
    pages = {}
    seen = {seed}
    queue = deque([seed])
    while queue:
        url = queue.popleft()
        status, html = miniweb.fetch(url)
        if status != 200:
            continue
        pages[url] = html
        for link in extract_links(url, html):
            if link not in seen:
                seen.add(link)
                queue.append(link)
    return pages

In [ ]:
pages = crawl("https://directory.example/")
print(len(pages), "pages crawled")

## Stage 4: Build the link graph

The crawl gives a dictionary from URL to HTML. Ranking needs a **graph**: for each page, the set of pages it links to. Write `build_graph(pages)`. It returns a dictionary that maps each crawled URL to a `set` of the crawled URLs it links to. Leave out:

- links to pages that are not in `pages` (they returned `404`), and
- links from a page to itself.

A page with no links out maps to an empty set; it still belongs in the graph.

`in_degree` is written for you. It counts the links pointing **into** each page: the simplest possible measure of importance.

In [ ]:
def build_graph(pages):
    graph = {}
    for url, html in pages.items():
        graph[url] = {link for link in extract_links(url, html)
                      if link in pages and link != url}
    return graph


def in_degree(graph):
    counts = {url: 0 for url in graph}
    for links in graph.values():
        for target in links:
            counts[target] += 1
    return counts

In [ ]:
graph = build_graph(pages)
counts = in_degree(graph)
for url in sorted(counts, key=counts.get, reverse=True)[:10]:
    print(counts[url], url)

Look at the ten pages with the most links pointing in. Several are not pages a person looking for health information should see first. Keep this list for the questions at the end.

## Stage 5: PageRank

Counting links treats every link as equal. PageRank does not: a link from an important page counts for more than a link from an obscure one, and a page is important when important pages link to it. The definition is circular, and it is resolved by repetition.

Picture a **random surfer**. On each step the surfer follows a random link on the current page, with probability `damping` (0.85), or jumps to a page chosen at random from the whole web, with probability `1 - damping`. A page's PageRank is the fraction of time the surfer spends on it.

The calculation starts with every page at rank `1 / n` and repeats one step at a time:

- Every page receives `(1 - damping) / n` from the random jumps.
- Each page with links out shares `damping * rank[page]` equally among the pages it links to.
- A page with no links out (a **dangling** page) has nowhere to send its rank, so it shares `damping * rank[page]` equally among **all** `n` pages.

The ranks always sum to 1. The loop stops when they stop changing.

Complete one step of the loop in `pagerank(graph)`.

In [ ]:
def pagerank(graph, damping=0.85, tolerance=1e-10, max_steps=200):
    n = len(graph)
    rank = {url: 1 / n for url in graph}
    for _ in range(max_steps):
        dangling_total = sum(rank[url] for url, links in graph.items() if not links)
        # Every page starts the step with its share of the random jumps and of the dangling pages.
        new_rank = {url: (1 - damping) / n + damping * dangling_total / n for url in graph}
        for page, links in graph.items():
            if links:
                share = damping * rank[page] / len(links)
                for target in links:
                    new_rank[target] += share
        change = sum(abs(new_rank[url] - rank[url]) for url in graph)
        rank = new_rank
        if change < tolerance:
            break
    return rank

In [ ]:
ranks = pagerank(graph)
print("total rank:", round(sum(ranks.values()), 6))
for url in sorted(ranks, key=ranks.get, reverse=True)[:10]:
    print(round(ranks[url], 4), url)

## Stage 6: Search

A search engine does not read every page when a query arrives. It reads every page once, in advance, and builds an **inverted index**: for each word, the set of pages that contain it. A query then only looks up its words.

`tokenize` is written for you. It lower-cases text and splits it into words.

Write `build_index(pages)`. It returns a dictionary from each word to the `set` of URLs whose text (from `extract_text`) contains that word.

Then write `search(query, index, scores, k=10)`. It returns a list of at most `k` URLs: the pages that contain **every** word of the query, highest score first. When two pages have the same score, put them in alphabetical order of URL. An empty query returns an empty list.

`scores` can be any dictionary from URL to a number, so the same function can rank by PageRank or by link count.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def build_index(pages):
    index = {}
    for url, html in pages.items():
        for word in tokenize(extract_text(html)):
            index.setdefault(word, set()).add(url)
    return index


def search(query, index, scores, k=10):
    words = tokenize(query)
    if not words:
        return []
    hits = set(index.get(words[0], set()))
    for word in words[1:]:
        hits &= index.get(word, set())
    return sorted(hits, key=lambda url: (-scores[url], url))[:k]

In [ ]:
index = build_index(pages)
print("By link count:")
for url in search("flu", index, counts, k=5):
    print("  ", url)
print("By PageRank:")
for url in search("flu", index, ranks, k=5):
    print("  ", url)

## Questions

Answer each question in two to four sentences, using output from your notebook.

**1.** Compare the ten pages with the most links pointing in (Stage 4) with the ten pages with the highest PageRank (Stage 5). Which pages appear in the first list but not the second, and why does PageRank rank them lower?

**2.** Run `pagerank(graph, damping=0.5)` and `pagerank(graph, damping=0.95)`, and add up the rank held by the pages on the three "cure" sites in each case. How does the damping factor change the total, and why? Consider where rank goes once the random surfer enters those sites.

**3.** Searching for `flu` by PageRank puts the hospital's visiting-hours page above the public health flu page. Why does this happen, and what would a search engine need besides PageRank to fix it?

*Answer 1:*

*Answer 2:*

*Answer 3:*